In [ ]:
from pathlib import Path
import cv2
import pandas as pd
import torch


DATA_DIR = Path(r"C:\Users\Toranj\Desktop\AIM\project2\nexar_data")
CSV_FILE = DATA_DIR / "metadata.csv"
VIDEO_DIR = DATA_DIR / "videos"
SAVE_DIR = DATA_DIR / "model_output"

          
FRAME_INTERVAL = 0.5       


SAVE_DIR.mkdir(parents=True, exist_ok=True)
FRAME_DIR = FRAMES_DIR = Path(r"C:\Users\Toranj\Desktop\AIM\project2\test\data\AIM_new_3lc\frames")


ID_COL = "video_id"
EVENT_COL = "time_of_event"
INDEX_COL = "dataset_index"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 5050 Laptop GPU


In [ ]:
df = pd.read_csv(CSV_FILE)

df["video_id"] = df[ID_COL].astype(str)
df["label"] = (pd.to_numeric(df[EVENT_COL], errors="coerce").notna().astype(int))
df["frame_dir"] = df[INDEX_COL].apply(lambda i: FRAMES_DIR / f"video_{int(i):06d}")
df = df[df["frame_dir"].apply(lambda p: p.exists() and len(list(p.glob("*.jpg"))) > 0)].copy()
df = df.drop_duplicates("video_id").reset_index(drop=True)


### looogs
print("Videos with extracted frames:", len(df))
print("\nClass distribution:")
print(df["label"].value_counts())

df[["video_id", "dataset_index", "label", "frame_dir"]].head()

Videos with extracted frames: 1500

Class distribution:
label
0    750
1    750
Name: count, dtype: int64


,video_id,dataset_index,label,frame_dir
0,video_000000,0,0,C:\Users\Toranj\Desktop\AIM\project2\test\data...
1,video_000001,1,0,C:\Users\Toranj\Desktop\AIM\project2\test\data...
2,video_000002,2,0,C:\Users\Toranj\Desktop\AIM\project2\test\data...
3,video_000003,3,0,C:\Users\Toranj\Desktop\AIM\project2\test\data...
4,video_000004,4,0,C:\Users\Toranj\Desktop\AIM\project2\test\data...


In [ ]:
#### new sampling / updated 

def extract_frames(video_path, output_folder):
    output_folder.mkdir(parents=True, exist_ok=True)

    # skiping videos that were already extracted
    existing = sorted(output_folder.glob("*.jpg"))
    if len(existing) > 0:
        return

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        cap.release()
        raise ValueError(f"Invalid FPS: {video_path}")

    frame_step = max(1, int(fps * FRAME_INTERVAL))
    frame_number = 0
    saved_number = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        if frame_number % frame_step == 0:
            frame = cv2.resize(frame, (224, 224))
            cv2.imwrite(str(output_folder / f"{saved_number:05d}.jpg"), frame)
            saved_number += 1

        frame_number += 1

    cap.release()
    if saved_number == 0:
        raise ValueError(f"No frames extracted: {video_path}")



for _, row in df.iterrows():
    extract_frames(Path(row["video_path"]), FRAME_DIR / row["video_id"])

print("Done")